In [ ]:
from pathlib import Path
import pandas as pd

print("=" * 75)
print("STEP 11 — SAVE ORIGINAL 4,132-GENE ENRICHMENT INPUT")
print("=" * 75)

OUTDIR = Path(
    "/content/drive/MyDrive/rp_network/data/processed/"
    "pathway_enrichment_4132"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

# Exact unique STRING preferred names from Stage 4
gene_list = sorted(unique_genes)

# TXT: one gene per line — convenient for enrichment tools
TXT_FILE = OUTDIR / "02_ppi_nodes_4132_gene_symbols.txt"

with open(TXT_FILE, "w") as f:
    for gene in gene_list:
        f.write(gene + "\n")

# CSV: permanent tabular record
CSV_FILE = OUTDIR / "02_ppi_nodes_4132_gene_symbols.csv"

pd.DataFrame({
    "gene_symbol": gene_list
}).to_csv(
    CSV_FILE,
    index=False
)

# QC
print("\nGenes saved:", len(gene_list))
print("Unique genes:", len(set(gene_list)))
print("Blank entries:", sum(g.strip() == "" for g in gene_list))

print("\nTXT:")
print(TXT_FILE)

print("\nCSV:")
print(CSV_FILE)

print("\nFirst 10:")
print("\n".join(gene_list[:10]))

print("\nLast 10:")
print("\n".join(gene_list[-10:]))

assert len(gene_list) == 4132
assert len(set(gene_list)) == 4132
assert all(g.strip() for g in gene_list)

print("\n✓ Exact 4,132-name Stage 4 network input preserved.")
print("=" * 75)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

project_dir = Path(
    '/content/drive/MyDrive/rp_network/data/processed/pathway_enrichment_4132'
)

project_dir.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_dir)
print("\nExisting files:")

for file in sorted(project_dir.iterdir()):
    print(file.name)

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nFiles uploaded:")
for filename in uploaded:
    print(filename)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

# Our permanent Google Drive folder
output_dir = Path(
    "/content/drive/MyDrive/rp_network/data/processed/pathway_enrichment_4132"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Original Enrichr exports
filenames = [
    "Reactome_Pathways_2024_table.txt",
    "KEGG_2026_table.txt"
]

for filename in filenames:
    source = Path("/content") / filename
    destination = output_dir / filename

    if not source.exists():
        print(f"ERROR: File not found: {filename}")
        continue

    # Preserve the original downloaded file
    if destination.exists():
        print(f"Already exists, not overwritten: {filename}")
    else:
        shutil.copy2(source, destination)
        print(f"Saved to Google Drive: {filename}")

    # Read the saved enrichment table
    df = pd.read_csv(destination, sep="\t")

    print(f"Number of pathway results: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    print("-" * 60)

In [ ]:
import pandas as pd

folder = "/content/drive/MyDrive/rp_network/data/processed/pathway_enrichment_4132"

reactome = pd.read_csv(
    f"{folder}/Reactome_Pathways_2024_table.txt",
    sep="\t"
)

kegg = pd.read_csv(
    f"{folder}/KEGG_2026_table.txt",
    sep="\t"
)

for name, df in [("Reactome", reactome), ("KEGG", kegg)]:
    print(f"\n===== {name} =====")
    print("Pathway rows:", len(df))
    print("Missing pathway names:", df["Term"].isna().sum())
    print("Missing gene lists:", df["Genes"].isna().sum())
    print("Duplicate pathway names:", df["Term"].duplicated().sum())
    print("First pathway:", df["Term"].iloc[0])
    print("Last pathway:", df["Term"].iloc[-1])
    print("First pathway gene-list example:")
    print(str(df["Genes"].iloc[0])[:250])

In [ ]:
from pathlib import Path
import pandas as pd

output_dir = Path(
    "/content/drive/MyDrive/rp_network/data/processed/pathway_enrichment_4132"
)

for database, df in [
    ("Reactome_2024", reactome),
    ("KEGG_2026", kegg)
]:
    # Preserve all exported enrichment results
    summary = df.copy()

    summary["Database"] = database

    summary["Participating_Gene_Count"] = (
        summary["Genes"]
        .str.split(";")
        .apply(len)
    )

    summary["Significant_FDR_0_05"] = (
        summary["Adjusted P-value"] < 0.05
    )

    summary_file = (
        output_dir / f"03_{database}_pathway_summary.csv"
    )

    summary.to_csv(summary_file, index=False)

    # Create one row for each pathway–gene relationship
    mapping = summary[
        ["Database", "Term", "Genes"]
    ].copy()

    mapping["Genes"] = mapping["Genes"].str.split(";")

    mapping = mapping.explode("Genes")

    mapping = mapping.rename(columns={
        "Term": "Pathway",
        "Genes": "Gene"
    })

    mapping["Gene"] = mapping["Gene"].str.strip()

    mapping_file = (
        output_dir / f"04_{database}_pathway_gene_mapping.csv"
    )

    mapping.to_csv(mapping_file, index=False)

    print(f"\n===== {database} =====")
    print("Pathway summary:", summary_file.name)
    print("Pathway rows:", len(summary))
    print(
        "Significant pathways (adjusted P < 0.05):",
        summary["Significant_FDR_0_05"].sum()
    )
    print("Pathway-gene mapping:", mapping_file.name)
    print("Pathway-gene relationships:", len(mapping))
    print("Unique genes:", mapping["Gene"].nunique())
    print("-" * 55)

In [ ]:
import pandas as pd

databases = ["Reactome_2024", "KEGG_2026"]

for database in databases:

    summary = pd.read_csv(
        output_dir / f"03_{database}_pathway_summary.csv"
    )

    mapping = pd.read_csv(
        output_dir / f"04_{database}_pathway_gene_mapping.csv"
    )

    expected_links = summary["Participating_Gene_Count"].sum()

    actual_links = len(mapping)

    missing_genes = mapping["Gene"].isna().sum()

    duplicate_links = mapping.duplicated(
        subset=["Database", "Pathway", "Gene"]
    ).sum()

    print(f"\n===== {database} =====")
    print("Saved pathways:", len(summary))
    print("Expected pathway-gene links:", expected_links)
    print("Actual pathway-gene links:", actual_links)
    print("Missing genes:", missing_genes)
    print("Duplicate pathway-gene links:", duplicate_links)

    assert expected_links == actual_links
    assert missing_genes == 0
    assert duplicate_links == 0

    print("RESULT: PASSED")

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

output_dir = Path(
    "/content/drive/MyDrive/rp_network/data/processed/pathway_enrichment_4132"
)

record_path = output_dir / "05_pathway_enrichment_methodology.md"

record = f"""# Pathway Enrichment — 4,132 STRING Network Proteins

## Documentation date
{datetime.now(ZoneInfo("Europe/Vilnius")).strftime("%Y-%m-%d")}

## Objective
Identify biological pathways enriched among proteins in the
retinitis pigmentosa (RP) protein-protein interaction network
and preserve the input genes associated with each pathway.

## Input
- Source: Stage 4 STRING protein-protein interaction network.
- Input file: 02_ppi_nodes_4132_gene_symbols.txt
- Unique input identifiers: 4,132.
- Organism: Homo sapiens.
- STRING interaction confidence threshold: 0.400.
- Input: unique protein names from both network endpoints.
- This enrichment uses the full Stage 4 network, not the
  smaller Stage 5 retinal-context-filtered network.

## Enrichment tool
- Tool: Enrichr
- Website: https://maayanlab.cloud/Enrichr/
- Libraries:
  - Reactome Pathways 2024
  - KEGG 2026
- Input submitted: 4,132 identifiers.
- Original enrichment exports preserved unchanged.
- Background: Enrichr's default library-specific statistical
  background; no custom background was supplied.
- Significance criterion: adjusted P-value < 0.05.
- Adjusted P-values were taken directly from Enrichr.

## Results
### Reactome Pathways 2024
- Exported pathways: 1,976.
- Significant pathways: 809.
- Unique participating input genes: 3,011.
- Pathway-gene relationships: 45,486.

### KEGG 2026
- Exported pathways: 349.
- Significant pathways: 227.
- Unique participating input genes: 2,284.
- Pathway-gene relationships: 13,531.

## Quality control
- No missing pathway names.
- No missing pathway gene lists.
- No duplicate pathway names within either export.
- All reported participating genes match the original input.
- All pathway overlap counts match their gene-list lengths.
- No missing genes in the expanded mapping tables.
- No duplicate pathway-gene relationships.
- Expanded mapping counts agree with summary gene counts.

## Files
### Input
- 01_enrichment_input_identifier_audit.tsv
- 02_ppi_nodes_4132_gene_symbols.csv
- 02_ppi_nodes_4132_gene_symbols.txt

### Unmodified Enrichr exports
- Reactome_Pathways_2024_table.txt
- KEGG_2026_table.txt

### Analysis-ready files
- 03_Reactome_2024_pathway_summary.csv
- 03_KEGG_2026_pathway_summary.csv
- 04_Reactome_2024_pathway_gene_mapping.csv
- 04_KEGG_2026_pathway_gene_mapping.csv

## Interpretation and limitations
- Reported pathway genes are overlaps with the submitted
  protein list, not complete pathway membership lists.
- Significant enrichment does not prove causal involvement
  in RP or apoptosis.
- The full network contains interaction partners beyond
  established RP disease genes.
- Gene recognition by Enrichr has not been independently
  confirmed for every submitted identifier.
- Further pathway interpretation and network reconstruction
  require separate analyses.
"""

if record_path.exists():
    print("Methodology file already exists; not overwritten.")
else:
    record_path.write_text(record, encoding="utf-8")
    print("Methodology record saved successfully.")

print("File:", record_path)
print("Size:", record_path.stat().st_size, "bytes")

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(filename)

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

# Permanent pathway enrichment folder
output_dir = Path(
    "/content/drive/MyDrive/rp_network/data/processed/pathway_enrichment_4132"
)

# Uploaded ToppFun file
source = Path(
    "/content/cfe657b9-ed0a-498f-8eb9-52a6a36687de.txt"
)

# Permanent filename
destination = output_dir / "06_ToppFun_original_results.txt"

# Preserve the original export
if not source.exists():
    raise FileNotFoundError(f"Uploaded file not found: {source}")

if destination.exists():
    print("Original file already saved. Not overwritten.")
else:
    shutil.copy2(source, destination)
    print("Original ToppFun file saved successfully.")

# Read the saved results
toppfun = pd.read_csv(destination, sep="\t")

print("\n===== TOPPFUN RESULTS =====")
print("Total pathway rows:", len(toppfun))
print("\nColumns:")
print(toppfun.columns.tolist())

print("\nResults by pathway source:")
print(toppfun["Source"].value_counts())

print("\nMissing pathway names:", toppfun["Name"].isna().sum())
print("Missing gene lists:", toppfun["Hit in Query List"].isna().sum())

print("\nSaved location:")
print(destination)

In [ ]:
import pandas as pd

# Create the pathway summary
toppfun_summary = toppfun.copy()

toppfun_summary["Significant_FDR_0_05"] = (
    toppfun_summary["q-value FDR B&H"] < 0.05
)

toppfun_summary["Participating_Gene_Count"] = (
    toppfun_summary["Hit in Query List"]
    .str.split(",")
    .apply(len)
)

summary_path = output_dir / "07_ToppFun_pathway_summary.csv"
toppfun_summary.to_csv(summary_path, index=False)

# Create the pathway-to-gene mapping
toppfun_mapping = toppfun_summary[
    ["ID", "Name", "Source", "Hit in Query List"]
].copy()

toppfun_mapping["Hit in Query List"] = (
    toppfun_mapping["Hit in Query List"].str.split(",")
)

toppfun_mapping = toppfun_mapping.explode("Hit in Query List")

toppfun_mapping = toppfun_mapping.rename(columns={
    "ID": "Pathway_ID",
    "Name": "Pathway",
    "Hit in Query List": "Gene"
})

toppfun_mapping["Gene"] = toppfun_mapping["Gene"].str.strip()

mapping_path = output_dir / "08_ToppFun_pathway_gene_mapping.csv"
toppfun_mapping.to_csv(mapping_path, index=False)

# Verify the outputs
print("===== TOPPFUN OUTPUTS =====")
print("Pathway summary rows:", len(toppfun_summary))
print("Pathway-gene mapping rows:", len(toppfun_mapping))
print("Unique participating genes:", toppfun_mapping["Gene"].nunique())

print("\nQuality control:")
print(
    "Gene-count mismatches:",
    (
        toppfun_summary["Participating_Gene_Count"]
        != toppfun_summary["Hit Count in Query List"]
    ).sum()
)
print("Missing genes:", toppfun_mapping["Gene"].isna().sum())
print(
    "Duplicate pathway-gene relationships:",
    toppfun_mapping.duplicated(
        subset=["Pathway_ID", "Source", "Gene"]
    ).sum()
)

print("\nSaved files:")
print(summary_path.name)
print(mapping_path.name)

In [ ]:
import pandas as pd

# Load saved pathway-gene mappings
enrichr_reactome = pd.read_csv(
    output_dir / "04_Reactome_2024_pathway_gene_mapping.csv"
)

enrichr_kegg = pd.read_csv(
    output_dir / "04_KEGG_2026_pathway_gene_mapping.csv"
)

toppfun_map = pd.read_csv(
    output_dir / "08_ToppFun_pathway_gene_mapping.csv"
)

# Unique genes in each tool's exported pathways
enrichr_genes = set(
    enrichr_reactome["Gene"].dropna().str.upper()
) | set(
    enrichr_kegg["Gene"].dropna().str.upper()
)

toppfun_genes = set(
    toppfun_map["Gene"].dropna().str.upper()
)

shared = enrichr_genes & toppfun_genes
enrichr_only = enrichr_genes - toppfun_genes
toppfun_only = toppfun_genes - enrichr_genes

print("===== ENRICHR VS TOPPFUN =====")
print("Enrichr unique participating genes:", len(enrichr_genes))
print("ToppFun unique participating genes:", len(toppfun_genes))
print("Genes shared by both tools:", len(shared))
print("Genes found only in Enrichr results:", len(enrichr_only))
print("Genes found only in ToppFun results:", len(toppfun_only))

print("\nExamples of Enrichr-only genes:")
print(sorted(enrichr_only)[:20])

print("\nExamples of ToppFun-only genes:")
print(sorted(toppfun_only)[:20])

In [ ]:
import pandas as pd
import re

# Load pathway summaries
enrichr_reactome_summary = pd.read_csv(
    output_dir / "03_Reactome_2024_pathway_summary.csv"
)

toppfun_summary = pd.read_csv(
    output_dir / "07_ToppFun_pathway_summary.csv"
)

# Keep statistically significant Reactome pathways
er_sig = enrichr_reactome_summary[
    enrichr_reactome_summary["Adjusted P-value"] < 0.05
].copy()

tf_sig = toppfun_summary[
    (toppfun_summary["Source"] == "Reactome Pathways")
    & (toppfun_summary["q-value FDR B&H"] < 0.05)
].copy()

# Normalize pathway names for an initial comparison
def normalize_pathway(name):
    name = str(name).upper()
    name = re.sub(r"^REACTOME_", "", name)
    name = re.sub(r"[^A-Z0-9]+", "_", name)
    return name.strip("_")

er_sig["Normalized_Pathway"] = (
    er_sig["Term"].apply(normalize_pathway)
)

tf_sig["Normalized_Pathway"] = (
    tf_sig["Name"].apply(normalize_pathway)
)

er_names = set(er_sig["Normalized_Pathway"])
tf_names = set(tf_sig["Normalized_Pathway"])

shared_pathways = er_names & tf_names

print("===== SIGNIFICANT REACTOME COMPARISON =====")
print("Enrichr significant pathway rows:", len(er_sig))
print("Enrichr unique normalized pathways:", len(er_names))
print("ToppFun significant pathway rows:", len(tf_sig))
print("ToppFun unique normalized pathways:", len(tf_names))
print("Shared normalized pathway names:", len(shared_pathways))
print("Enrichr-only normalized names:", len(er_names - tf_names))
print("ToppFun-only normalized names:", len(tf_names - er_names))

print("\nExamples of shared pathways:")
for pathway in sorted(shared_pathways)[:20]:
    print(pathway)

In [ ]:
# Build comparison of significant normalized Reactome pathway names
all_names = sorted(er_names | tf_names)

comparison = pd.DataFrame({
    "Normalized_Pathway": all_names
})

comparison["Enrichr_Significant"] = (
    comparison["Normalized_Pathway"].isin(er_names)
)

comparison["ToppFun_Significant"] = (
    comparison["Normalized_Pathway"].isin(tf_names)
)

comparison["Agreement"] = comparison.apply(
    lambda row: (
        "Both"
        if row["Enrichr_Significant"] and row["ToppFun_Significant"]
        else "Enrichr_only"
        if row["Enrichr_Significant"]
        else "ToppFun_only"
    ),
    axis=1
)

# Attach Enrichr pathway names and FDR values
er_details = er_sig[
    ["Normalized_Pathway", "Term", "Adjusted P-value"]
].rename(columns={
    "Term": "Enrichr_Pathway",
    "Adjusted P-value": "Enrichr_FDR"
})

comparison = comparison.merge(
    er_details,
    on="Normalized_Pathway",
    how="left",
    validate="one_to_one"
)

# Aggregate ToppFun records without discarding distinct IDs
tf_details = (
    tf_sig.groupby("Normalized_Pathway")
    .agg(
        ToppFun_Pathway=("Name", "first"),
        ToppFun_IDs=("ID", lambda x: ";".join(x.astype(str))),
        ToppFun_Record_Count=("ID", "size"),
        ToppFun_Best_FDR=("q-value FDR B&H", "min")
    )
    .reset_index()
)

comparison = comparison.merge(
    tf_details,
    on="Normalized_Pathway",
    how="left",
    validate="one_to_one"
)

# Save comparison
comparison_path = (
    output_dir / "09_Reactome_Enrichr_ToppFun_comparison.csv"
)

comparison.to_csv(comparison_path, index=False)

print("===== REACTOME COMPARISON SAVED =====")
print("Total unique pathway names:", len(comparison))

print("\nAgreement categories:")
print(comparison["Agreement"].value_counts())

print("\nToppFun records retained through IDs:")
print(int(comparison["ToppFun_Record_Count"].sum()))

print("\nSaved file:")
print(comparison_path)

In [ ]:
from pathlib import Path
from datetime import date

methodology = f"""# ToppFun Pathway Enrichment and Reactome Comparison

Date documented: {date.today().isoformat()}

## 1. Research objective
Identify annotated biological pathways associated with the
4,132 proteins in the Stage 4 STRING protein interaction network
and retain the genes participating in each pathway.

## 2. Input dataset
Input: 4,132 unique gene symbols from the Stage 4 network.
The original input list was preserved without automatic remapping.

## 3. ToppFun enrichment
Tool: ToppFun, ToppGene Suite.
Website: https://toppgene.cchmc.org/

Input identifiers: HGNC gene symbols.
Submitted identifiers: 4,132.
Recognized identifiers: 4,047.
Unrecognized identifiers: 85.

Pathway-category annotated input genes: 3,499.
Pathway annotations before applied cutoff: 3,697.
Reference genes in pathway category: 14,250.

Selected pathway sources:
- Reactome Pathways
- KEGG Legacy Pathways
- KEGG Medicus Pathways

Background: ToppFun default category-specific reference.
Statistical correction: FDR Benjamini-Hochberg.
Significance cutoff: FDR < 0.05.
Gene limits: 1 to 2,000.

## 4. ToppFun exported results
Total significant exported pathway records: 1,306.
Reactome: 967.
KEGG Legacy: 86.
KEGG Medicus: 253.

Total pathway-gene relationships: 46,576.
Unique genes in exported pathway results: 3,034.

Quality control:
- Gene-count mismatches: 0.
- Missing participating genes: 0.
- Duplicate pathway-ID/source/gene relationships: 0.

Original export:
06_ToppFun_original_results.txt

Derived files:
07_ToppFun_pathway_summary.csv
08_ToppFun_pathway_gene_mapping.csv

## 5. Enrichr comparison
Enrichr pathway libraries:
- Reactome Pathways 2024
- KEGG 2026

Enrichr unique participating genes across exported results: 3,239.
ToppFun unique participating genes across exported results: 3,034.
Shared participating genes: 3,004.
Enrichr-only genes: 235.
ToppFun-only genes: 30.

These gene comparisons concern exported pathway memberships,
not necessarily significant gene-level associations.

## 6. Significant Reactome pathway comparison
Enrichr significant Reactome records: 809.
ToppFun significant Reactome records: 967.
ToppFun unique normalized pathway names: 608.

Shared significant normalized pathway names: 577.
Enrichr-only normalized names: 232.
ToppFun-only normalized names: 31.

Total distinct significant normalized names: 840.

Comparison file:
09_Reactome_Enrichr_ToppFun_comparison.csv

## 7. Important methodological limitations
1. Enrichr and ToppFun use different pathway annotation versions.
2. Their background populations and identifier recognition may differ.
3. Enrichr exports include nonsignificant pathways; the ToppFun
   export contains pathways passing the selected FDR cutoff.
4. Normalized pathway-name agreement does not guarantee identical
   pathway definitions or participating gene sets.
5. ToppFun records with distinct IDs were preserved even when
   pathway names were identical.
6. No causal relationship between RP genes and apoptosis can be
   inferred from enrichment alone.
7. The 85 unrecognized ToppFun identifiers require separate
   documentation and possible investigation.

## 8. Next research stage
Review pathway results with the thesis supervisor before selecting
specific pathways or conducting further network reconstruction.
"""

methodology_path = (
    output_dir / "10_ToppFun_Reactome_comparison_methodology.md"
)

methodology_path.write_text(methodology, encoding="utf-8")

print("===== METHODOLOGY DOCUMENT SAVED =====")
print("File:", methodology_path)
print("Characters:", len(methodology))
print("File exists:", methodology_path.exists())

In [ ]:
import pandas as pd
import re

# Load saved summaries
enrichr_kegg_summary = pd.read_csv(
    output_dir / "03_KEGG_2026_pathway_summary.csv"
)

toppfun_summary = pd.read_csv(
    output_dir / "07_ToppFun_pathway_summary.csv"
)

# Significant Enrichr KEGG pathways
ek_sig = enrichr_kegg_summary[
    enrichr_kegg_summary["Adjusted P-value"] < 0.05
].copy()

# Significant ToppFun KEGG pathways
tk_sig = toppfun_summary[
    toppfun_summary["Source"].isin([
        "KEGG Legacy Pathways",
        "KEGG Medicus Pathways"
    ])
    & (toppfun_summary["q-value FDR B&H"] < 0.05)
].copy()

# Normalize names
def normalize_kegg(name):
    name = str(name).upper()
    name = re.sub(r"^KEGG_", "", name)
    name = re.sub(r"[^A-Z0-9]+", "_", name)
    return name.strip("_")

ek_sig["Normalized_Pathway"] = (
    ek_sig["Term"].apply(normalize_kegg)
)

tk_sig["Normalized_Pathway"] = (
    tk_sig["Name"].apply(normalize_kegg)
)

enrichr_names = set(ek_sig["Normalized_Pathway"])

print("===== KEGG PATHWAY COMPARISON =====")
print("Enrichr KEGG significant pathways:", len(ek_sig))

for source in [
    "KEGG Legacy Pathways",
    "KEGG Medicus Pathways"
]:
    subset = tk_sig[tk_sig["Source"] == source]

    toppfun_names = set(subset["Normalized_Pathway"])

    print(f"\n--- {source} ---")
    print("Significant pathway records:", len(subset))
    print("Unique normalized names:", len(toppfun_names))
    print("Shared with Enrichr KEGG:", len(enrichr_names & toppfun_names))
    print("Enrichr-only names:", len(enrichr_names - toppfun_names))
    print("ToppFun-only names:", len(toppfun_names - enrichr_names))

    print("\nExamples of shared pathways:")
    for pathway in sorted(enrichr_names & toppfun_names)[:10]:
        print(pathway)

In [ ]:
import pandas as pd

files = {
    "Enrichr Reactome": "04_Reactome_2024_pathway_gene_mapping.csv",
    "Enrichr KEGG": "04_KEGG_2026_pathway_gene_mapping.csv",
    "ToppFun": "08_ToppFun_pathway_gene_mapping.csv"
}

for label, filename in files.items():
    df = pd.read_csv(output_dir / filename)

    print(f"\n===== {label} =====")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))

In [ ]:
import pandas as pd
import re

def normalize_pathway(name):
    name = str(name).upper().strip()
    name = re.sub(r"^REACTOME_", "", name)
    name = re.sub(r"[^A-Z0-9]+", "_", name)
    return name.strip("_")

# Load saved pathway-gene mappings
er = pd.read_csv(
    output_dir / "04_Reactome_2024_pathway_gene_mapping.csv"
)

tf = pd.read_csv(
    output_dir / "08_ToppFun_pathway_gene_mapping.csv"
)

# Load the previously saved significant pathway comparison
comparison = pd.read_csv(
    output_dir / "09_Reactome_Enrichr_ToppFun_comparison.csv"
)

# Identify the 577 shared significant pathway names
shared_names = set(
    comparison.loc[
        comparison["Agreement"] == "Both",
        "Normalized_Pathway"
    ]
)

# Normalize pathway names
er["Normalized_Pathway"] = er["Pathway"].apply(normalize_pathway)
tf["Normalized_Pathway"] = tf["Pathway"].apply(normalize_pathway)

# Keep Reactome ToppFun records only
tf = tf[tf["Source"] == "Reactome Pathways"].copy()

# Prepare Enrichr gene sets
er_sets = (
    er[er["Normalized_Pathway"].isin(shared_names)]
    .groupby("Normalized_Pathway")["Gene"]
    .apply(lambda x: set(x.dropna().astype(str).str.strip()))
    .to_dict()
)

# Compare each ToppFun pathway ID separately
results = []

tf_shared = tf[
    tf["Normalized_Pathway"].isin(shared_names)
]

for (name, pathway_id), group in tf_shared.groupby(
    ["Normalized_Pathway", "Pathway_ID"]
):
    enrichr_genes = er_sets.get(name, set())

    toppfun_genes = set(
        group["Gene"].dropna().astype(str).str.strip()
    )

    shared = enrichr_genes & toppfun_genes
    enrichr_only = enrichr_genes - toppfun_genes
    toppfun_only = toppfun_genes - enrichr_genes
    union = enrichr_genes | toppfun_genes

    results.append({
        "Normalized_Pathway": name,
        "ToppFun_Pathway_ID": pathway_id,
        "Enrichr_Gene_Count": len(enrichr_genes),
        "ToppFun_Gene_Count": len(toppfun_genes),
        "Shared_Gene_Count": len(shared),
        "Enrichr_Only_Gene_Count": len(enrichr_only),
        "ToppFun_Only_Gene_Count": len(toppfun_only),
        "Jaccard_Similarity": len(shared) / len(union) if union else 0,
        "Shared_Genes": ";".join(sorted(shared)),
        "Enrichr_Only_Genes": ";".join(sorted(enrichr_only)),
        "ToppFun_Only_Genes": ";".join(sorted(toppfun_only))
    })

reactome_gene_comparison = pd.DataFrame(results)

# Save without modifying original files
save_path = (
    output_dir / "13_Reactome_gene_membership_comparison.csv"
)

reactome_gene_comparison.to_csv(save_path, index=False)

print("===== REACTOME GENE-MEMBERSHIP COMPARISON =====")
print("Shared significant pathway names:", len(shared_names))
print("Compared ToppFun pathway IDs:", len(reactome_gene_comparison))
print(
    "Unique matched pathway names:",
    reactome_gene_comparison["Normalized_Pathway"].nunique()
)

print(
    "Mean Jaccard similarity:",
    round(reactome_gene_comparison["Jaccard_Similarity"].mean(), 4)
)

print(
    "Median Jaccard similarity:",
    round(reactome_gene_comparison["Jaccard_Similarity"].median(), 4)
)

print(
    "Identical gene sets:",
    (reactome_gene_comparison["Jaccard_Similarity"] == 1).sum()
)

print("\nSaved:", save_path)

In [ ]:
# STEP 24 — KEGG Legacy gene-membership comparison

import pandas as pd
import re

def normalize_kegg_gene_comparison(name):
    name = str(name).upper().strip()
    name = re.sub(r"^KEGG_", "", name)
    name = re.sub(r"[^A-Z0-9]+", "_", name)
    return name.strip("_")

ek = pd.read_csv(
    output_dir / "04_KEGG_2026_pathway_gene_mapping.csv"
)

tf = pd.read_csv(
    output_dir / "08_ToppFun_pathway_gene_mapping.csv"
)

kc = pd.read_csv(
    output_dir / "11_KEGG_Enrichr_ToppFun_comparison.csv"
)

shared_kegg_names = set(
    kc.loc[
        (kc["ToppFun_Source"] == "KEGG Legacy Pathways")
        & (kc["Agreement"] == "Both"),
        "Normalized_Pathway"
    ]
)

ek["Normalized_Pathway"] = ek["Pathway"].apply(
    normalize_kegg_gene_comparison
)

tf["Normalized_Pathway"] = tf["Pathway"].apply(
    normalize_kegg_gene_comparison
)

tf_legacy = tf[
    (tf["Source"] == "KEGG Legacy Pathways")
    & (tf["Normalized_Pathway"].isin(shared_kegg_names))
].copy()

ek_sets = (
    ek[ek["Normalized_Pathway"].isin(shared_kegg_names)]
    .groupby("Normalized_Pathway")["Gene"]
    .apply(lambda x: set(x.dropna().astype(str).str.strip()))
    .to_dict()
)

results = []

for (name, pathway_id), group in tf_legacy.groupby(
    ["Normalized_Pathway", "Pathway_ID"]
):
    enrichr_genes = ek_sets.get(name, set())
    toppfun_genes = set(
        group["Gene"].dropna().astype(str).str.strip()
    )

    shared = enrichr_genes & toppfun_genes
    enrichr_only = enrichr_genes - toppfun_genes
    toppfun_only = toppfun_genes - enrichr_genes
    union = enrichr_genes | toppfun_genes

    results.append({
        "Normalized_Pathway": name,
        "ToppFun_Pathway_ID": pathway_id,
        "Enrichr_Gene_Count": len(enrichr_genes),
        "ToppFun_Gene_Count": len(toppfun_genes),
        "Shared_Gene_Count": len(shared),
        "Enrichr_Only_Gene_Count": len(enrichr_only),
        "ToppFun_Only_Gene_Count": len(toppfun_only),
        "Jaccard_Similarity": (
            len(shared) / len(union) if union else 0
        ),
        "Shared_Genes": ";".join(sorted(shared)),
        "Enrichr_Only_Genes": ";".join(sorted(enrichr_only)),
        "ToppFun_Only_Genes": ";".join(sorted(toppfun_only))
    })

kegg_gene_comparison = pd.DataFrame(results)

save_path = (
    output_dir / "14_KEGG_Legacy_gene_membership_comparison.csv"
)

kegg_gene_comparison.to_csv(save_path, index=False)

print("===== KEGG LEGACY GENE-MEMBERSHIP COMPARISON =====")
print("Shared significant pathway names:", len(shared_kegg_names))
print("Compared ToppFun pathway IDs:", len(kegg_gene_comparison))
print(
    "Unique matched pathway names:",
    kegg_gene_comparison["Normalized_Pathway"].nunique()
)
print(
    "Mean Jaccard similarity:",
    round(kegg_gene_comparison["Jaccard_Similarity"].mean(), 4)
)
print(
    "Median Jaccard similarity:",
    round(kegg_gene_comparison["Jaccard_Similarity"].median(), 4)
)
print(
    "Identical gene sets:",
    (kegg_gene_comparison["Jaccard_Similarity"] == 1).sum()
)

print("\nSaved:", save_path)

In [ ]:
from pathlib import Path

report = """# Pathway Gene-Membership Comparison

## Objective
Compare participating genes reported by Enrichr and ToppFun
for significant pathways with matching normalized names.

## Reactome
Shared significant pathway names: 577
ToppFun pathway-ID comparisons: 926
Mean Jaccard similarity: 0.9279
Median Jaccard similarity: 1.0000
Identical reported gene sets: 478

Output:
13_Reactome_gene_membership_comparison.csv

## KEGG Legacy
Shared significant pathway names: 78
ToppFun pathway-ID comparisons: 78
Mean Jaccard similarity: 0.7367
Median Jaccard similarity: 0.7588
Identical reported gene sets: 0

Output:
14_KEGG_Legacy_gene_membership_comparison.csv

## Method
Gene symbols were extracted from the saved pathway-gene
mapping tables.

For each matched normalized pathway name, gene sets were
compared using intersection, set difference, and Jaccard similarity.

Jaccard similarity = number of shared genes divided by
number of genes in the union of both sets.

Each ToppFun pathway ID was retained as a separate record.

## Important limitations
These results compare genes reported in the enrichment
outputs, not necessarily all genes annotated to each pathway.

Reactome contains multiple ToppFun IDs for some normalized
pathway names. Consequently, its mean Jaccard similarity
is calculated across pathway-ID comparisons rather than
independent pathway names.

Matching pathway names does not establish identical
underlying pathway definitions.

Differences in annotation versions, identifier recognition,
gene mapping, and enrichment settings may affect agreement.

KEGG Medicus was not included in gene-membership comparisons
because no exact normalized pathway-name matches were found.

The results do not establish biological causality or
independent experimental validation.
"""

path = output_dir / "15_gene_membership_comparison_methodology.md"

path.write_text(report, encoding="utf-8")

print("===== GENE-MEMBERSHIP METHODOLOGY SAVED =====")
print("File:", path)
print("Exists:", path.exists())
print("Characters:", len(report))

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("===== SEARCHING GOOGLE DRIVE FOR NOTEBOOK =====")

matches = []

for file in drive_root.rglob("*.ipynb"):
    if (
        "Pathway" in file.name
        or "4132" in file.name
        or "Enrichment" in file.name
    ):
        matches.append(file)
        print("FOUND:", file)

print("\nTotal matching notebooks:", len(matches))

if not matches:
    print(
        "No matching notebook found in Google Drive. "
        "We may need to save the current Colab notebook to Drive."
    )